# `json2vec` Hello World

This notebook trains a tiny model from an in-memory synthetic dataset.

In [12]:
import lightning.pytorch as lit
import torch
from rich.pretty import pprint

import json2vec as j2v

In [13]:
@j2v.shim(yields=True)
def hello_world_records(observation: dict, strata: j2v.Strata):

    records = [
        {"color": "red", "label": "warm"},
        {"color": "orange", "label": "warm"},
        {"color": "yellow", "label": "warm"},
        {"color": "blue", "label": "cool"},
        {"color": "green", "label": "cool"},
        {"color": "purple", "label": "cool"},
        {"color": "red", "label": "warm"},
        {"color": "blue", "label": "cool"},
    ]

    yield from records

In [14]:
params = j2v.Hyperparameters(
    d_model=16,
    target=j2v.Address("record", "label"),
    embed=j2v.Address("record"),
    fields=j2v.Array(
        name="record",
        fields=[
            j2v.Category(name="color", query="[*].color", max_vocab_size=16),
            j2v.Category(name="label", query="[*].label", max_vocab_size=8, topk=[2]),
        ],
    ),
)


model = j2v.JSON2Vec(
    hyperparameters=params,
    batch_size=4,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.StreamingDataModule.from_model(
    model,
    dataset=j2v.Dataset(processor=hello_world_records),
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    file_buffer_size=1,
    observation_buffer_size=32,
    sample_rate=1.0,
)

2026-05-17 19:42:51.221 | INFO     | json2vec.architecture.root:__init__:128 - initialized JSON2Vec module


In [15]:
trainer = lit.Trainer(
    accelerator="cpu",
    max_epochs=20,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False
)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
2026-05-17 19:42:51.238 | INFO     | json2vec.data.datasets:dataloader:670 - building dataloader
2026-05-17 19:42:51.239 | INFO     | json2vec.data.datasets:__init__:614 - initialized batch dataset
2026-05-17 19:42:51.250 | INFO     | json2vec.data.datasets:dataloader:670 - building dataloader
2026-05-17 19:42:51.250 | INFO     | json2vec.data.datasets:__init__:614 - initialized batch dataset
`Trainer.fit` stopped: `max_epochs=20` reached.


In [16]:
batch = [[{"color": "red"}], [{"color": "blue"}]]

In [17]:
pprint(model.predict(batch))

{
│   'record/label': {
│   │   'state': {
│   │   │   'valued': [0.9946720004081726, 0.995582103729248],
│   │   │   'null': [0.0010569111909717321, 0.000992780551314354],
│   │   │   'padded': [0.0019109838176518679, 0.002003858331590891],
│   │   │   'masked': [0.0012172788847237825, 0.0006556017324328423],
│   │   │   'other': [0.0011427629506215453, 0.000765733071602881]
│   │   },
│   │   'content': {
│   │   │   'value': ['warm', 'cool'],
│   │   │   'probability': [0.9191471338272095, 0.9843889474868774],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'warm', 'probability': 0.9191471338272095},
│   │   │   │   │   {'label': 'cool', 'probability': 0.08085278421640396}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'cool', 'probability': 0.9843889474868774},
│   │   │   │   │   {'label': 'warm', 'probability': 0.015610977075994015}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

In [18]:
pprint(model.embed(batch))

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   0.2123042494058609,
│   │   │   │   0.5615366101264954,
│   │   │   │   -0.36969780921936035,
│   │   │   │   -0.2893051207065582,
│   │   │   │   -0.39806100726127625,
│   │   │   │   -0.1994437575340271,
│   │   │   │   0.0893912985920906,
│   │   │   │   0.07658199965953827,
│   │   │   │   0.2497294694185257,
│   │   │   │   0.22958719730377197,
│   │   │   │   -0.2532683312892914,
│   │   │   │   0.1264885514974594,
│   │   │   │   -0.038874585181474686,
│   │   │   │   0.03829854726791382,
│   │   │   │   -0.06493678689002991,
│   │   │   │   0.06877192854881287
│   │   │   ],
│   │   │   [
│   │   │   │   -0.43186596035957336,
│   │   │   │   -0.14655925333499908,
│   │   │   │   0.24880610406398773,
│   │   │   │   0.3630792796611786,
│   │   │   │   0.001085073803551495,
│   │   │   │   0.36146852374076843,
│   │   │   │   -0.07445301115512848,
│   │   │   │   -0.2031720131635666,
│   │   │   │   -0.2666226923465729,
│   │   │   │   -0.22641777992248535,
│   │   │   │   0.06078875809907913,
│   │   │   │   0.3977455794811249,
│   │   │   │   -0.2677769362926483,
│   │   │   │   0.18513603508472443,
│   │   │   │   0.11199470609426498,
│   │   │   │   -0.1342613399028778
│   │   │   ]
│   │   ]
│   }
}